# Supplementary Measures — Education Attendance, Awareness Sources, Income Allocation, Financial Goals, and Three Planned Relationships

Continues from `docs/research_and_measurement_plan.md`, `docs/analysis_coverage_checklist.md` (items 11-13, "needs calculation"), and `docs/survey_schema_reference.md`. This notebook computes every measure identified as supported-but-not-yet-calculated in the previous pass:

1. Awareness-source and media distributions (`Q4_Q5_NONInv_Filt[{_1_2}].Q4M`/`.Q5M`) for the focused group.
2. Corrected income-allocation distributions (`Q1MXGrid[{_1..5}].Q1M`, the true raw field — `Q1M_DP` is a derived field that silently converts "not administered" into a `"0%"` category, confirmed by direct comparison below).
3. Financial-goal ranking (`Q6_RANK_GRID`, 13 slots) — rank meaning and eligibility verified beforehand (all 553 have >=1 non-blank slot; every pattern is a top-3 priority ranking).
4. An independent recheck of `Q12M` and `Q20AM` against the already-exported `demographics.json` figures.
5. The three planned relationships: `QRT` x fear-of-loss, a pre-registered knowledge item (`GRIDxQ15AM[{_1}]`, chosen for being the one item specifically about mutual-fund product mechanics, not generic finance or account/KYC mechanics) x "better education," and `GRIDxQ15AM[{_4}]` (online KYC) x "simple process."

Scope: unweighted, descriptive only. No significance tests, no causal claims, no invented scores. No respondent-level data in any output below. The reason-field tokenizer and the small-group reporting rule (`SMALL_GROUP_MIN_N = 30`) are reused from `03_descriptive_analysis.ipynb`/`04_segment_comparisons.ipynb`, not redefined.

In [1]:
import itertools
from collections import Counter
from pathlib import Path

import pandas as pd
from python_calamine import CalamineWorkbook

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
ANALYSIS_TABLE_DIR = PROC_DIR / "analysis"
ANALYSIS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

SMALL_GROUP_MIN_N = 30  # fixed, identical to 04_segment_comparisons.ipynb -- not redefined per comparison

## 1. Load the corrected extracts and re-verify cohort membership

In [2]:
broader = pd.read_csv(PROC_DIR / "cohort_salaried_genz_mains.csv", keep_default_na=False)
focused = pd.read_csv(PROC_DIR / "cohort_focused_considered_mf_not_holding.csv", keep_default_na=False)

print("broader:", broader.shape, " focused:", focused.shape)
assert len(broader) == 4346, f"broader group count changed: {len(broader)} != 4346"
assert len(focused) == 553, f"focused group count changed: {len(focused)} != 553"

broader: (4346, 20)  focused: (553, 30)


## 2. Reload the raw workbook and join the columns not present in either extract

Same pattern as the income join in `04_segment_comparisons.ipynb`: reload raw, merge onto `focused` by `Resp_ID_DP`, assert no unexpected nulls. `Resp_ID_DP` is used only as a join key and is never displayed or exported.

In [3]:
wb = CalamineWorkbook.from_path(str(RAW_DIR / "Respondent Data.XLSX"))
sheet = wb.get_sheet_by_name(wb.sheet_names[0])
data = sheet.to_python(skip_empty_area=True)
raw_codes, raw_descs, raw_rows = data[0], data[1], data[2:]
raw_df = pd.DataFrame(raw_rows, columns=raw_codes)
assert raw_df.shape == (109430, 448)
print("raw_df:", raw_df.shape)

raw_df: (109430, 448)


In [4]:
QRT_COL = "QRT"
GRID_COLS = [f"GRIDxQ15AM[{{_{i}}}].Q15AM" for i in range(1, 10)]
Q1MX_COLS = [f"Q1MXGrid[{{_{i}}}].Q1M" for i in range(1, 6)]
GOAL_COLS = [f"Q6_RANK_GRID[{{_{i}}}].Q6_RANK" for i in range(1, 13)] + ["Q6_RANK_GRID[{_19}].Q6_RANK"]
AWARE_Q4M = "Q4_Q5_NONInv_Filt[{_1_2}].Q4M"
AWARE_Q5M = "Q4_Q5_NONInv_Filt[{_1_2}].Q5M"

join_cols = ["Resp_ID_DP", QRT_COL, "Q20AM", "Q12M", AWARE_Q4M, AWARE_Q5M] + GRID_COLS + Q1MX_COLS + GOAL_COLS
focused = focused.merge(raw_df[join_cols], on="Resp_ID_DP", how="left")
assert focused[QRT_COL].isna().sum() == 0
assert focused["Q20AM"].isna().sum() == 0
assert focused["Q12M"].isna().sum() == 0
print("joined shape:", focused.shape)

joined shape: (553, 62)


## 3. Reason-field tokenizer for `AA2_DD2`/`AA3_DD3` -- reuse the already-verified vocabulary

`derive_reason_vocab` (from `03_descriptive_analysis.ipynb`) is not perfectly deterministic across re-runs when several fragments recur an equal number of times in the promotion step (a scratch re-run produced 19/11 options instead of the verified 18/10). Rather than re-derive it, this notebook loads the **already-saved, previously-validated** vocabulary directly from `data/processed/analysis/barrier_option_counts.csv` and reuses only the (deterministic) `resolve_reason` resolver against it — per the instruction to reuse completed, validated work rather than duplicate it.

In [5]:
def resolve_reason(v, atomic, expected_items=3):
    frags = [f.strip() for f in v.split(",")]
    tokens, i = [], 0
    while i < len(frags):
        matched = False
        for j in range(len(frags), i, -1):
            candidate = ", ".join(frags[i:j])
            if candidate in atomic:
                tokens.append(candidate)
                i = j
                matched = True
                break
        if not matched:
            return None
    return tokens if len(tokens) == expected_items else None


_barrier_option_counts = pd.read_csv(ANALYSIS_TABLE_DIR / "barrier_option_counts.csv")
reason_vocabs = {
    col: set(_barrier_option_counts.loc[_barrier_option_counts["field"] == col, "option"])
    for col in ["AA2_DD2", "AA3_DD3"]
}
assert len(reason_vocabs["AA2_DD2"]) == 18
assert len(reason_vocabs["AA3_DD3"]) == 10
print({k: len(v) for k, v in reason_vocabs.items()})

{'AA2_DD2': 18, 'AA3_DD3': 10}


In [6]:
FEAR_OF_LOSS = "Fear of losing money due to market risks"
BETTER_EDUCATION = "Better education on how mutual funds work"
SIMPLE_PROCESS = "Simple and easy process for investing (e.g. account opening, documentation, etc.)"
assert FEAR_OF_LOSS in reason_vocabs["AA2_DD2"]
assert BETTER_EDUCATION in reason_vocabs["AA3_DD3"]
assert SIMPLE_PROCESS in reason_vocabs["AA3_DD3"]


def selected(raw_value, option, vocab):
    if raw_value == "":
        return None  # missing -- kept distinct from "did not select"
    toks = resolve_reason(raw_value, vocab)
    assert toks is not None, f"unresolved: {raw_value!r}"
    return option in toks


focused["selected_fear_of_loss"] = focused["AA2_DD2_raw"].apply(lambda v: selected(v, FEAR_OF_LOSS, reason_vocabs["AA2_DD2"]))
focused["selected_better_education"] = focused["AA3_DD3_raw"].apply(lambda v: selected(v, BETTER_EDUCATION, reason_vocabs["AA3_DD3"]))
focused["selected_simple_process"] = focused["AA3_DD3_raw"].apply(lambda v: selected(v, SIMPLE_PROCESS, reason_vocabs["AA3_DD3"]))

n_aa2_answered = int(focused["selected_fear_of_loss"].notna().sum())
n_aa3_answered = int(focused["selected_better_education"].notna().sum())
print("AA2_DD2 substantive answerers:", n_aa2_answered, "(expect 266)")
print("AA3_DD3 substantive answerers:", n_aa3_answered, "(expect 266)")
assert n_aa2_answered == 266
assert n_aa3_answered == 266

AA2_DD2 substantive answerers: 266 (expect 266)
AA3_DD3 substantive answerers: 266 (expect 266)


## 4. Awareness-source and media distributions (`Q4_Q5_NONInv_Filt[{_1_2}]`)

Multi-select, with several option labels containing internal commas (e.g. "Friends, Family, and Colleagues"). The atomic vocabulary below was derived by inspecting all 18,223 workbook-wide non-holder respondents (the full population this field is routed to) and verified to fully resolve **100% of those 18,223 raw values with zero unresolved fragments** using the same protect-and-split technique already used for `Q20CM`/`Q20DM`/`Q20F` in `scripts/export_demographics_psychographics.py:227-244` (`tokenize(value, vocabulary)`, copied here verbatim).

Routing check: `Q4_Q5_NONInv_Filt[{_1_2}].Q4M`/`.Q5M` are answered by **exactly** the same 18,223 workbook-wide respondents as `AA2_DD2` -- confirmed by exact respondent-ID-set comparison. Within the focused group, this is verified below to be the identical 266 who answer `AA2_DD2`. Only a combined MF+ETF slot exists for non-holders (no MF-only slot) -- this is a property of the survey design, not a scope choice made here.

In [7]:
Q4M_VOCAB = [
    "Friends, Family, and Colleagues",
    "Financial Influencers on social media (YouTube, Instagram, Facebook, LinkedIn, Twitter)",
    "Financial Professionals - Financial advisors/planners, bank representatives",
    "Financial News & Blogs",
    "Advertisements on investment products",
    "Market or company analysis reports (Brokerage firm reports, industry whitepapers, company financial reports)",
    "Online Investment Communities (Telegram groups, WhatsApp groups, Reddit, Facebook groups)",
    "Educational Resources (Investment webinars, courses, books, podcasts, financial literacy workshops)",
    "Investor Education Programmes run by different investment companies (broking houses etc.)",
    "Investor Education Programmes run by prominent institutions/industry associations (SEBI, NISM, Exchanges, Depositories, AMFI/Mutual fund sahi hai etc)",
    "None of the above",
    "Others (please specify)",
]
Q5M_VOCAB = [
    "Television",
    "Others (please specify)",
    "In person Consultations / Meetings",
    "Other online media",
    "Online Search Engines",
    "Online Newspapers/Magazines/Publications",
    "Phone calls / SMS",
    "Physical Newspapers/Magazines/Publications",
    "Fin-Tech Apps & Investment Platforms",
    "Radio",
    "Social media (YouTube videos, Instagram reels, Twitter/X posts)",
    "Messaging apps (WhatsApp, telegram, etc.)",
    "Websites/Apps of regulators (like AMFI, SEBI , etc.)",
    "Seminars, Webinars, and Workshops",
]


def tokenize(value, vocabulary):
    """Protect known comma-bearing option labels, split on the remaining commas, then
    restore -- same technique as scripts/export_demographics_psychographics.py:227-244.
    Raises if any fragment doesn't resolve into the known vocabulary."""
    protected = value
    placeholders = {}
    for i, opt in enumerate(vocabulary):
        if "," in opt:
            token = f"\x00{i}\x00"
            placeholders[token] = opt
            protected = protected.replace(opt, token)
    parts = [p.strip() for p in protected.split(",")]
    resolved = [placeholders.get(p, p) for p in parts]
    unresolved = [p for p in resolved if p not in vocabulary]
    if unresolved:
        raise ValueError(f"Unresolved fragment(s) {unresolved!r} in value {value!r}")
    return resolved


# Verify the vocabulary against the FULL workbook-wide non-holder population (18,223) before
# using it on the focused group -- this is the population the field is actually routed to.
_full_q4 = raw_df.loc[raw_df[AWARE_Q4M] != "", AWARE_Q4M]
_full_q5 = raw_df.loc[raw_df[AWARE_Q5M] != "", AWARE_Q5M]
assert len(_full_q4) == 18223 and len(_full_q5) == 18223
_bad_q4 = sum(1 for v in _full_q4 if (lambda: (tokenize(v, Q4M_VOCAB), False)[1] if True else True))
for v in _full_q4:
    tokenize(v, Q4M_VOCAB)  # raises on any unresolved fragment -- confirms 100% resolution
for v in _full_q5:
    tokenize(v, Q5M_VOCAB)
print("Q4M/Q5M vocabularies fully resolve all 18,223 workbook-wide non-holder respondents.")

Q4M/Q5M vocabularies fully resolve all 18,223 workbook-wide non-holder respondents.


In [8]:
aware_q4_ids = set(focused.loc[focused[AWARE_Q4M] != "", "Resp_ID_DP"])
aware_q5_ids = set(focused.loc[focused[AWARE_Q5M] != "", "Resp_ID_DP"])
aa2_ids = set(focused.loc[focused["AA2_DD2_raw"] != "", "Resp_ID_DP"])
print("awareness Q4M answered (focused):", len(aware_q4_ids), "(expect 266)")
print("awareness Q5M answered (focused):", len(aware_q5_ids), "(expect 266)")
assert len(aware_q4_ids) == 266 and len(aware_q5_ids) == 266
assert aware_q4_ids == aa2_ids == aware_q5_ids, "awareness answerers are not the identical AA2_DD2 answer set"
print("Confirmed: identical 266-respondent answer base as AA2_DD2.")

awareness Q4M answered (focused): 266 (expect 266)
awareness Q5M answered (focused): 266 (expect 266)
Confirmed: identical 266-respondent answer base as AA2_DD2.


In [9]:
def option_counts_multiselect(series, vocabulary):
    nonblank = series[series != ""]
    exploded = []
    for v in nonblank:
        exploded.extend(tokenize(v, vocabulary))
    counts = pd.Series(exploded).value_counts()
    n = len(nonblank)
    tbl = pd.DataFrame({
        "option": counts.index, "n": counts.values,
        "pct_of_answered": (100 * counts.values / n).round(1),
    }).sort_values("n", ascending=False)
    tbl["denominator"] = n
    return tbl


sources_tbl = option_counts_multiselect(focused[AWARE_Q4M], Q4M_VOCAB)
sources_tbl.insert(0, "field", "sources")
media_tbl = option_counts_multiselect(focused[AWARE_Q5M], Q5M_VOCAB)
media_tbl.insert(0, "field", "media")

print("\n=== Awareness sources (Q4M), denominator = 266 ===")
print(sources_tbl.drop(columns="field").to_string(index=False))
print("\n=== Awareness media (Q5M), denominator = 266 ===")
print(media_tbl.drop(columns="field").to_string(index=False))

awareness_sources_media = pd.concat([sources_tbl, media_tbl], ignore_index=True)
awareness_sources_media.to_csv(ANALYSIS_TABLE_DIR / "awareness_sources_media.csv", index=False)
print("\nwrote", ANALYSIS_TABLE_DIR / "awareness_sources_media.csv")


=== Awareness sources (Q4M), denominator = 266 ===
                                                                                                                                                option   n  pct_of_answered  denominator
                                                                                                                       Friends, Family, and Colleagues 154             57.9          266
                                                               Financial Influencers on social media (YouTube, Instagram, Facebook, LinkedIn, Twitter) 144             54.1          266
                                                             Online Investment Communities (Telegram groups, WhatsApp groups, Reddit, Facebook groups)  79             29.7          266
                                                                           Financial Professionals - Financial advisors/planners, bank representatives  70             26.3          266
                       

## 5. Corrected income-allocation distributions

`Q1M_DP[{_1..5}].Q1M` is a **derived** field: it bands the raw `Q1MXGrid[{_1..5}].Q1M` percentage into decile ranges, but wherever the raw field is blank, `Q1M_DP` hard-sets the value to the bare number `0.0` (displayed as `"0%"`) -- confirmed directly below by comparing raw-blank counts to `Q1M_DP`'s `"0%"` counts, which differ substantially (e.g. slot 4/Investments: 32 raw-blank vs 75 shown as `"0%"` in the derived field). This recomputes all 5 distributions from the true raw field, keeping blank as blank.

In [10]:
def band(x):
    if x == 0.0:
        return "0%"
    lo = int((x - 1) // 10) * 10 + 1
    hi = lo + 9
    return f"{lo}-{hi}%"


INCOME_ALLOC_LABELS = {
    1: "Monthly expenses", 2: "Savings", 3: "Loan repayments",
    4: "Investments", 5: "Other expenses",
}

income_alloc_rows = []
for i in range(1, 6):
    col = f"Q1MXGrid[{{_{i}}}].Q1M"
    vals = focused[col]
    is_blank = vals.isna() | (vals == "")
    n_blank = int(is_blank.sum())
    nonblank = vals[~is_blank].astype(float)
    n_answered = len(nonblank)
    counts = nonblank.apply(band).value_counts()
    for opt, n in counts.items():
        income_alloc_rows.append({
            "slot": i, "category": INCOME_ALLOC_LABELS[i], "option": opt, "n": int(n),
            "pct_of_answered": round(100 * n / n_answered, 1),
            "denominator": n_answered, "n_blank": n_blank,
        })
    print(f"slot {i} ({INCOME_ALLOC_LABELS[i]}): n_answered={n_answered}, n_blank={n_blank}")

income_allocation_corrected = pd.DataFrame(income_alloc_rows)
income_allocation_corrected.to_csv(ANALYSIS_TABLE_DIR / "income_allocation_corrected.csv", index=False)
print("\nwrote", ANALYSIS_TABLE_DIR / "income_allocation_corrected.csv")

slot 1 (Monthly expenses): n_answered=534, n_blank=19
slot 2 (Savings): n_answered=532, n_blank=21
slot 3 (Loan repayments): n_answered=517, n_blank=36
slot 4 (Investments): n_answered=521, n_blank=32
slot 5 (Other expenses): n_answered=527, n_blank=26

wrote ../data/processed/analysis/income_allocation_corrected.csv


## 6. Financial-goal ranking (`Q6_RANK_GRID`)

Eligibility and rank meaning verified beforehand: all 553 focused-group respondents have at least one non-blank slot among the 12 named goals or the free-text 13th ("Others") slot, and every respondent's non-blank pattern is exactly a top-3 priority ranking (values `{1.0, 2.0, 3.0}`). This reports "selected in top 3" (any non-blank rank) per goal, the same selection-percentage convention already used for `AA1_DD1`-style measures -- not a rank-weighted score.

In [11]:
GOAL_LABELS = {
    1: "Buying a house", 2: "Children's education", 3: "Planning for retirement",
    4: "Building an emergency fund", 5: "Growing wealth", 6: "Saving for a major expense",
    7: "Generating passive income", 8: "Supporting family members", 9: "Achieving financial independence",
    10: "My child's marriage", 11: "Optimizing tax savings and benefits", 12: "Earning money actively on a daily basis",
}

def is_blank_val(v):
    return v is None or v == "" or (isinstance(v, float) and pd.isna(v))

any_named_goal = pd.Series(False, index=focused.index)
for i in range(1, 13):
    col = f"Q6_RANK_GRID[{{_{i}}}].Q6_RANK"
    any_named_goal = any_named_goal | (~focused[col].apply(is_blank_val))
used_others = ~focused["Q6_RANK_GRID[{_19}].Q6_RANK"].apply(is_blank_val)
n_any_goal = int((any_named_goal | used_others).sum())
n_used_others = int(used_others.sum())
print("focused respondents with >=1 non-blank goal rank:", n_any_goal, "(expect 553)")
print("focused respondents who used the free-text 'Others' 13th slot:", n_used_others)
assert n_any_goal == 553

goal_rows = []
for i in range(1, 13):
    col = f"Q6_RANK_GRID[{{_{i}}}].Q6_RANK"
    n_ranked = int((~focused[col].apply(is_blank_val)).sum())
    goal_rows.append({"goal": GOAL_LABELS[i], "n_ranked_in_top3": n_ranked, "pct_of_553": round(100 * n_ranked / 553, 1)})
goal_rows.append({"goal": "Others (free text)", "n_ranked_in_top3": n_used_others, "pct_of_553": round(100 * n_used_others / 553, 1)})

financial_goal_ranking = pd.DataFrame(goal_rows).sort_values("n_ranked_in_top3", ascending=False)
print(financial_goal_ranking.to_string(index=False))
financial_goal_ranking.to_csv(ANALYSIS_TABLE_DIR / "financial_goal_ranking.csv", index=False)
print("\nwrote", ANALYSIS_TABLE_DIR / "financial_goal_ranking.csv")

focused respondents with >=1 non-blank goal rank: 553 (expect 553)
focused respondents who used the free-text 'Others' 13th slot: 0
                                   goal  n_ranked_in_top3  pct_of_553
                         Growing wealth               235        42.5
              Supporting family members               206        37.3
             Building an emergency fund               196        35.4
                         Buying a house               194        35.1
       Achieving financial independence               159        28.8
             Saving for a major expense               128        23.1
                   Children's education               127        23.0
                Planning for retirement               108        19.5
Earning money actively on a daily basis                97        17.5
              Generating passive income                86        15.6
                    My child's marriage                73        13.2
    Optimizing tax savings a

## 7. Independent recheck of `Q12M` and `Q20AM` (already exported, verifying not recomputing-for-export)

In [12]:
q12m_counts = focused["Q12M"].value_counts()
print("Q12M n_blank:", int((focused["Q12M"] == "").sum()))
print(q12m_counts)
q12m_recheck = q12m_counts.reset_index()
q12m_recheck.columns = ["option", "n"]
q12m_recheck.to_csv(ANALYSIS_TABLE_DIR / "q12m_recheck.csv", index=False)

q20am_counts = focused["Q20AM"].value_counts()
print("\nQ20AM n_blank:", int((focused["Q20AM"] == "").sum()))
print(q20am_counts)
assert int(q20am_counts.get("Have not attended any investor education program", 0)) == 527

Q12M n_blank: 0
Q12M
Less than today     245
Exactly as today    130
More than today      94
Do not know          69
Refuse to answer     15
Name: count, dtype: int64

Q20AM n_blank: 0
Q20AM
Have not attended any investor education program                  527
Yes, attended it online (webinars / virtual training sessions)     20
Yes, attended it in-person (seminars / workshops)                   6
Name: count, dtype: int64


## 8. Three planned relationships

Each restricted to the substantive answerers of the relevant question (266 for both `AA2_DD2` and `AA3_DD3`), gated by the same `SMALL_GROUP_MIN_N = 30` rule already used for every other comparison in this project. Missing is kept distinct from "did not select" (there are no blanks in `QRT`/`GRIDxQ15AM`, so this only matters for the `AA2_DD2`/`AA3_DD3` selection flags, already computed as `None` for non-answerers above). No significance test. These describe associations observed in this sample, not causes.

In [13]:
def relationship_table(sub, group_col, flag_col, label):
    # n_selecting is the raw (unrounded-later) count selecting the target option within
    # each category -- saved alongside the already-rounded pct so any percentage-point
    # difference can be computed from raw counts rather than from two rounded percentages.
    rows = []
    for cat, g in sub.groupby(group_col):
        n = len(g)
        meets_min = n >= SMALL_GROUP_MIN_N
        n_selecting = int(g[flag_col].sum())
        pct = round(100 * n_selecting / n, 1) if meets_min else None
        rows.append({"category": cat, "n": n, "n_selecting": n_selecting, "meets_small_group_min": meets_min, f"pct_{label}": pct})
    tbl = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
    overall = round(100 * sub[flag_col].sum() / len(sub), 1)
    return tbl, overall


sub_a = focused[focused["selected_fear_of_loss"].notna()].copy()
rel_a, overall_a = relationship_table(sub_a, QRT_COL, "selected_fear_of_loss", "selecting_fear_of_loss")
print("=== (a) QRT x fear-of-loss -- n=266, overall selecting fear-of-loss:", overall_a, "% ===")
print(rel_a.to_string(index=False))
rel_a.to_csv(ANALYSIS_TABLE_DIR / "relationship_qrt_fear_of_loss.csv", index=False)

=== (a) QRT x fear-of-loss -- n=266, overall selecting fear-of-loss: 30.5 % ===
                                                                                                                                                category   n  n_selecting  meets_small_group_min  pct_selecting_fear_of_loss
                                                                                 I need to have good but stable and reliable returns with minimal losses 100           28                   True                        28.0
                                                                         Preservation of capital (amount invested) is more important to me than returns.  96           27                   True                        28.1
I aim for better, higher returns and realize that there will be some ups and downs in my investment, but I wouldn't be able to accept significant losses  48           17                   True                        35.4
       I would like high returns and

In [14]:
item1_col = "GRIDxQ15AM[{_1}].Q15AM"
sub_b = focused[focused["selected_better_education"].notna()].copy()
rel_b, overall_b = relationship_table(sub_b, item1_col, "selected_better_education", "selecting_better_education")
print("=== (b) GRIDxQ15AM[_1] (expense-ratio item) x better-education -- n=266, overall:", overall_b, "% ===")
print(rel_b.to_string(index=False))
rel_b.to_csv(ANALYSIS_TABLE_DIR / "relationship_knowledge_item1_education.csv", index=False)

=== (b) GRIDxQ15AM[_1] (expense-ratio item) x better-education -- n=266, overall: 36.5 % ===
 category   n  n_selecting  meets_small_group_min  pct_selecting_better_education
     TRUE 172           65                   True                            37.8
Not Aware  55           22                   True                            40.0
    FALSE  39           10                   True                            25.6


In [15]:
item4_col = "GRIDxQ15AM[{_4}].Q15AM"
sub_c = focused[focused["selected_simple_process"].notna()].copy()
rel_c, overall_c = relationship_table(sub_c, item4_col, "selected_simple_process", "selecting_simple_process")
print("=== (c) GRIDxQ15AM[_4] (online-KYC item) x simple-process -- n=266, overall:", overall_c, "% ===")
print(rel_c.to_string(index=False))
rel_c.to_csv(ANALYSIS_TABLE_DIR / "relationship_kyc_simple_process.csv", index=False)

=== (c) GRIDxQ15AM[_4] (online-KYC item) x simple-process -- n=266, overall: 44.0 % ===
 category   n  n_selecting  meets_small_group_min  pct_selecting_simple_process
     TRUE 208           94                   True                          45.2
    FALSE  30           13                   True                          43.3
Not Aware  28           10                  False                           NaN


## 9. Summary

- Awareness sources/media: same 266-respondent base as `AA2_DD2`; top source "Friends, Family, and Colleagues" (57.9%), top media "Social media" (57.5%).
- Income allocation, corrected: real blank rates now visible (e.g. 32-36 of 553 blank per slot for loan repayments/investments), no longer masked as "0%".
- Financial goals: "Growing wealth" (42.5%) and "Supporting family members" (37.3%) are the two most commonly top-3-ranked goals of 553; 0 focused-group respondents used the free-text "Others" slot.
- `Q12M`/`Q20AM`: reconfirmed, matches the already-exported `demographics.json` figures exactly.
- Relationship (a): weak, non-monotonic pattern -- the QRT category expressing *some* risk tolerance (35.4%) selects fear-of-loss slightly *more* than the two more conservative categories (28.0%/28.1%), not less. No clear risk-preference gradient.
- Relationship (b): weak/mixed -- respondents unsure about the expense-ratio fact ("Not Aware," 40.0%) select "better education" somewhat more than those who answered correctly (37.8%), but those who answered incorrectly ("FALSE," 25.6%) select it *least* -- not a consistent knowledge-gap pattern.
- Relationship (c): essentially no difference -- 45.2% (True) vs 43.3% (False) select "simple process," a 1.9pp gap; the "Not Aware" group is below the reporting minimum.

No respondent-level data is saved or printed anywhere above. All outputs are aggregate CSVs under `data/processed/analysis/`.